# 0. Collect official main-conference paper lists for selected CS conferences.
#### 从DBLP获取AAAI, ACL, CVPR, EMNLP, ICLR, ICML, IJCAI, KDD, SIGIR, WWW这些会议信息，注意区分主会和非主会

The output contains:
conference, year, title, doi, source_url, track, source, source_record_id,doi_source, section/track name

Design principle:
1. Use DBLP html (not API) proceedings sources wherever possible.
2. Treat DOI as optional because several official venues (ICLR, ICML, CVPR) often do not expose DOI values on their proceedings pages.
3. Keep source_url and notes for manual auditing.

notes: ACL, EMNLP, ICML, CVPR and ICLR这些会议可以从官方网站获取论文列表

## 1. DBLP获取AAAI主会文章
- 从DBLP的aaai/index.html页面获取aaai2020-2025的proceedings页面区分main proceedings和non-main proceedings
- 在main proceedings页面通过h2是否包含technical track/tracks来区分是否是main track

In [ ]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

AAAI_INDEX_URL = "https://dblp.org/db/conf/aaai/index.html"
YEARS = range(2020, 2026)

# All records collected from AAAI-related DBLP proceedings volumes.
OUT_CSV = OUT_DIR / "aaai_dblp_2020_2025_all_tracks.csv"


OUT_SUMMARY = OUT_DIR / "aaai_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# AAAI-specific rules
# =========================
NON_PAPER_TITLE_PATTERNS = [
    r"\bfront\s*matter\b",
    r"\bfrontmatter\b",
    r"\bpreface\b",
    r"\btable of contents\b",
    r"\bauthor index\b",
    r"\bprogram committee\b",
    r"\borganizing committee\b",
    r"\bproceedings of\b",
    r"\bwelcome message\b",
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_heading(text):
    return normalize_text(text).lower()


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
            return resp.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def is_non_paper_title(title):
    text = str(title or "").lower()
    return any(re.search(pattern, text) for pattern in NON_PAPER_TITLE_PATTERNS)


def is_aaai_main_h2(year, h2_text):
    h2 = normalize_heading(h2_text)

    if "iaai" in h2 or "eaai" in h2:
        return False

    if "special technical track" in h2:
        return False

    if year in {2020, 2022, 2023, 2024}:
        return h2.startswith("aaai technical track")

    if year in {2021, 2025}:
        return "technical tracks" in h2

    return False


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return normalize_doi_for_key(href.split("doi.org/", 1)[1]), "doi_link"
        if "doi.acm.org/" in href:
            return normalize_doi_for_key(href.rstrip("/").rsplit("/", 1)[-1]), "acm_doi_link"
    return "", ""


def extract_source_record_id(li):
    if li.get("id"):
        return li.get("id")

    for a in li.find_all("a", href=True):
        href = a["href"]
        if "/rec/" in href or "/pid/" in href:
            return href

    return ""


def extract_toc_url(li):
    for a in li.find_all("a", href=True):
        text = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(AAAI_INDEX_URL, a["href"])

        if "contents" in text and "/db/conf/aaai/" in href and href.endswith(".html"):
            return href

    return ""


def infer_aaai_year_from_dblp_key(dblp_key):
    tail = dblp_key.rsplit("/", 1)[-1]
    m = re.match(r"(20\d{2})", tail)
    if m:
        return int(m.group(1))
    return None


def infer_aaai_volume_type(dblp_key, proceedings_title):
    """
    Main AAAI proceedings use DBLP keys such as conf/aaai/2024.
    Extra volumes use suffixes such as conf/aaai/2024bridge or conf/aaai/2024ml4cmh.
    """
    tail = dblp_key.rsplit("/", 1)[-1]
    title = normalize_heading(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", "main_proceedings"

    if "bridge" in tail or "bridge" in title:
        return "non_main_proceedings", "bridge_program"

    if "safeai" in tail or "safeai" in title:
        return "non_main_proceedings", "safeai_workshop"

    if "ml4cmh" in tail or "machine learning for cognitive and mental health" in title:
        return "non_main_proceedings", "ml4cmh_workshop"

    if "workshop" in title:
        return "non_main_proceedings", "workshop"

    return "non_main_proceedings", "other_non_main_proceedings"


def collect_aaai_volumes_from_index():
    html_text = request_with_retry(AAAI_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    volumes = []

    for li in soup.select("li.entry"):
        dblp_key = li.get("id", "")
        if not dblp_key.startswith("conf/aaai/"):
            continue

        year = infer_aaai_year_from_dblp_key(dblp_key)
        if year not in YEARS:
            continue

        toc_url = extract_toc_url(li)
        if not toc_url:
            continue

        proceedings_title = extract_title(li)
        if not proceedings_title:
            proceedings_title = normalize_text(li.get_text(" ", strip=True))

        proceedings_type, proceedings_name = infer_aaai_volume_type(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        volumes.append({
            "conference": "AAAI",
            "year": year,
            "dblp_key": dblp_key,
            "contents_url": toc_url,
            "proceedings_title": proceedings_title,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
        })

    return pd.DataFrame(volumes)


def classify_paper_track(volume_row, current_h2):
    if volume_row["proceedings_type"] != "main_proceedings":
        return "non_main_track"

    if is_aaai_main_h2(int(volume_row["year"]), current_h2):
        return "main_track"

    return "non_main_track"


def collect_aaai_toc_page(volume_row):
    toc_url = volume_row["contents_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    current_h2 = ""

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            current_h2 = normalize_text(tag.get_text(" ", strip=True))
            continue

        if tag.name != "li":
            continue

        classes = tag.get("class", [])
        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(tag)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        authors = extract_authors(tag)
        doi, doi_source = extract_doi(tag)
        source_record_id = extract_source_record_id(tag)
        paper_track = classify_paper_track(volume_row, current_h2)

        is_main_proceedings = volume_row["proceedings_type"] == "main_proceedings"
        is_main_conference_paper = is_main_proceedings and paper_track == "main_track"

        rows.append({
            "conference": "AAAI",
            "year": int(volume_row["year"]),
            "title": title,
            "authors": authors,
            "doi": doi,
            "source_url": toc_url,
            "track_type": paper_track,
            "source": "dblp_aaai_index_and_h2_filter",
            "source_record_id": source_record_id,
            "doi_source": doi_source,
            "notes": (
                f"dblp_volume_key={volume_row['dblp_key']}; "
                f"proceedings_title={volume_row['proceedings_title']}; "
                f"proceedings_name={volume_row['proceedings_name']}; "
                f"h2={current_h2}; "
                f"index_url={AAAI_INDEX_URL}"
            ),
            "section/track_name": current_h2 if current_h2 else volume_row["proceedings_name"],
            "proceedings_type": volume_row["proceedings_type"],
            "is_main_proceedings": is_main_proceedings,
            "is_main_conference_paper": is_main_conference_paper,
        })

    return rows


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])




def standardize_output_schema(df):
    """Normalize DBLP collection output to the compact all-track schema."""
    df = df.copy()

    if "proceedings_type" not in df.columns:
        if "volume_track" in df.columns:
            df["proceedings_type"] = df["volume_track"]
        elif "is_main_proceedings" in df.columns:
            df["proceedings_type"] = df["is_main_proceedings"].map({True: "main_proceedings", False: "non-main_proceedings"})
        else:
            df["proceedings_type"] = ""

    if "proceedings_name" not in df.columns:
        if "proceedings_title" in df.columns:
            df["proceedings_name"] = df["proceedings_title"]
        elif "section/track_name" in df.columns:
            df["proceedings_name"] = df["section/track_name"]
        else:
            df["proceedings_name"] = ""

    if "track_type" not in df.columns:
        source_track = None
        for col in ["paper_track", "track"]:
            if col in df.columns:
                source_track = df[col].astype(str).str.lower()
                break

        if source_track is None:
            df["track_type"] = ""
        else:
            df["track_type"] = "non-main_track"
            main_mask = source_track.isin(["main_track", "main_research_track", "research_track", "main"])
            df.loc[main_mask, "track_type"] = "main_track"

    # Infer proceedings type from track/notes where it was not explicitly recorded.
    missing_proceedings = ~df["proceedings_type"].astype(str).isin(["main_proceedings", "non-main_proceedings", "non_main_proceedings"])
    if missing_proceedings.any():
        notes = df["notes"].astype(str).str.lower() if "notes" in df.columns else ""
        track = df["track_type"].astype(str)

        df.loc[missing_proceedings, "proceedings_type"] = "main_proceedings"

        if "notes" in df.columns:
            non_main_notes = notes.str.contains("volume_track=non_main_proceedings|volume_track=non-main_proceedings", regex=True)
            df.loc[non_main_notes, "proceedings_type"] = "non-main_proceedings"

        # For proceedings-level collectors such as ACL/EMNLP/ICLR/ICML, non-main track
        # usually means records came from a non-main proceedings volume.
        if "source" in df.columns:
            source = df["source"].astype(str).str.lower()
            proceedings_filter = source.str.contains("proceedings_filter", regex=False)
            df.loc[missing_proceedings & proceedings_filter & track.eq("non-main_track"), "proceedings_type"] = "non-main_proceedings"

    df["proceedings_type"] = (
        df["proceedings_type"]
        .astype(str)
        .str.replace("non_main_proceedings", "non-main_proceedings", regex=False)
        .str.replace("non-main-proceedings", "non-main_proceedings", regex=False)
    )
    df.loc[~df["proceedings_type"].isin(["main_proceedings", "non-main_proceedings"]), "proceedings_type"] = "main_proceedings"

    df["track_type"] = (
        df["track_type"]
        .astype(str)
        .str.replace("main_research_track", "main_track", regex=False)
        .str.replace("non_main_track", "non-main_track", regex=False)
        .str.replace("non-main-track", "non-main_track", regex=False)
    )
    df.loc[~df["track_type"].isin(["main_track", "non-main_track"]), "track_type"] = "non-main_track"

    if "section/track_name" not in df.columns:
        df["section/track_name"] = df["proceedings_name"]

    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    return df[STANDARD_COLUMNS]

# =========================
# Collect AAAI papers
# =========================
volumes_df = collect_aaai_volumes_from_index()

print("AAAI proceedings volumes found:")
display(
    volumes_df
    .groupby(["year", "proceedings_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "proceedings_name"])
)

all_rows = []

for _, row in volumes_df.sort_values(["year", "dblp_key"]).iterrows():
    print(
        f"Collecting AAAI {row['year']} | {row['proceedings_type']} | "
        f"{row['proceedings_name']} | {row['contents_url']}"
    )

    rows = collect_aaai_toc_page(row)
    print(f"  papers={len(rows)}")
    all_rows.extend(rows)

    time.sleep(random.uniform(0.6, 1.2))

aaai_all_df = pd.DataFrame(all_rows)

if aaai_all_df.empty:
    raise RuntimeError("No AAAI records were collected. Please inspect DBLP page structure.")

aaai_all_df = deduplicate(aaai_all_df)
aaai_all_df = aaai_all_df.reindex(columns=STANDARD_COLUMNS)
aaai_all_df = aaai_all_df.sort_values(
    ["year", "proceedings_type", "section/track_name", "title"]
).reset_index(drop=True)

summary = (
    aaai_all_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["year", "proceedings_type"])
)
summary.to_csv(OUT_SUMMARY, index=False)

aaai_all_df = standardize_output_schema(aaai_all_df)

aaai_all_df.to_csv(OUT_CSV, index=False)

print("\nSaved:")
print(f"  all tracks: {OUT_CSV}")
print(f"  summary: {OUT_SUMMARY}")


AAAI proceedings volumes found:


,year,proceedings_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_proceedings,1
1,2020,non_main_proceedings,safeai_workshop,1
2,2020,non_main_proceedings,workshop,1
3,2021,main_proceedings,main_proceedings,1
4,2021,non_main_proceedings,safeai_workshop,1
5,2021,non_main_proceedings,workshop,4
6,2022,main_proceedings,main_proceedings,1
7,2022,non_main_proceedings,safeai_workshop,1
8,2022,non_main_proceedings,workshop,1
9,2023,main_proceedings,main_proceedings,1


  papers=1864
  papers=17
  papers=24
  papers=1960
  papers=10
  papers=23
  papers=13
  papers=23
  papers=33
  papers=1623
  papers=27
  papers=30
  papers=2020
  papers=14
  papers=28
  papers=14
  papers=2865
  papers=2
  papers=15
  papers=3485
  papers=23

Saved:
  all tracks: ../Data/aaai_dblp_2020_2025_all_tracks.csv
  summary: ../Data/aaai_dblp_2020_2025_summary.csv


# 2. Collect ACL
- dblp acl/index.html页面中的proceedings列表可以直接区分出main proceedings/non main proceeedings和main track/non-main track

In [2]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ACL_INDEX_URL = "https://dblp.org/db/conf/acl/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "acl_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "acl_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(ACL_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/acl/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))
    match = re.search(r"(Proceedings of .*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text


def extract_acl_year_from_h2_tag(h2_tag):
    # First use the structural id of the h2 tag, if available.
    h2_id = normalize_text(h2_tag.get("id", ""))
    match = re.search(r"\b(20\d{2})\b", h2_id)
    if match:
        return int(match.group(1))

    return None


def classify_acl_proceedings(proceedings_title):
    """
    ACL is organized as separate proceedings volumes on DBLP.
    Main-track papers are identified at the proceedings-volume level.
    """
    proceedings_name = normalize_text(proceedings_title)
    t = proceedings_name.lower()

    non_main_patterns = [
        "findings of",
        "system demonstrations",
        "student research workshop",
        "tutorial abstracts",
        "industry track",
    ]

    if any(pattern in t for pattern in non_main_patterns):
        return "non-main_proceedings", proceedings_name, "non-main_track"

    main_patterns = [
        "long papers",
        "short papers",
        "annual meeting of the association for computational linguistics",# 2020年不区分long和short，所以是在判断了非main proceedings之后，剩下的包含这几个单词的就是main proceedings了
    ]

    if any(pattern in t for pattern in main_patterns):
        return "main_proceedings", proceedings_name, "main_track"

    return "non-main_proceedings", proceedings_name, "non-main_track"


def is_non_paper_title(title):
    text = str(title or "").lower()
    non_paper_patterns = [
        r"\bfront\s*matter\b",
        r"\bfrontmatter\b",
        r"\bpreface\b",
        r"\btable of contents\b",
        r"\bauthor index\b",
        r"\bprogram committee\b",
        r"\borganizing committee\b",
        r"\bproceedings of\b",
        r"\bwelcome message\b",
    ]
    return any(re.search(pattern, text) for pattern in non_paper_patterns)


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])


def collect_acl_proceedings_from_index():
    html_text = request_with_retry(ACL_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            year = extract_acl_year_from_h2_tag(tag)
            if year is not None:
                current_year = year
            continue

        if tag.name != "li" or current_year not in YEARS:
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)
        proceedings_type, proceedings_name, track_type = classify_acl_proceedings(proceedings_title)

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": proceedings_name,
        })

    return pd.DataFrame(proceedings_rows)


def collect_acl_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    for li in soup.find_all("li"):
        classes = li.get("class", [])
        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(li)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        rows.append({
            "conference": "ACL",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(li),
            "doi": extract_doi(li),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": proceedings_row["track_type"],
            "section/track_name": proceedings_row["section/track_name"],
        })

    return rows


# =========================
# Collect ACL papers from DBLP
# =========================
proceedings_df = collect_acl_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "track_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "track_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting ACL {row['year']} | {row['proceedings_type']} | "
        f"{row['track_type']} | {row['proceedings_name']}"
    )

    rows = collect_acl_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))

acl_df = pd.DataFrame(all_rows)

if acl_df.empty:
    raise RuntimeError("No ACL records were collected. Please inspect DBLP page structure.")

acl_df = deduplicate(acl_df)
acl_df = acl_df.reindex(columns=STANDARD_COLUMNS)
acl_df = acl_df.sort_values(
    ["year", "proceedings_type", "track_type", "proceedings_name", "title"]
).reset_index(drop=True)

acl_df.to_csv(OUT_CSV, index=False)

summary = (
    acl_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved ACL all-track papers to: {OUT_CSV}")
print(f"Saved ACL summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,track_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_track,Proceedings of the 58th Annual Meeting of the ...,1
1,2020,non-main_proceedings,non-main_track,Proceedings of the 58th Annual Meeting of the ...,1
2,2020,non-main_proceedings,non-main_track,Proceedings of the 58th Annual Meeting of the ...,1
3,2020,non-main_proceedings,non-main_track,Proceedings of the 58th Annual Meeting of the ...,1
4,2020,non-main_proceedings,non-main_track,Proceedings of the First Joint Workshop on Nar...,1
5,2021,main_proceedings,main_track,Proceedings of the 59th Annual Meeting of the ...,1
6,2021,main_proceedings,main_track,Proceedings of the 59th Annual Meeting of the ...,1
7,2021,non-main_proceedings,non-main_track,Findings of the Association for Computational ...,1
8,2021,non-main_proceedings,non-main_track,Proceedings of the ACL-IJCNLP 2021 Student Res...,1
9,2021,non-main_proceedings,non-main_track,Proceedings of the Joint Conference of the 59t...,1


  papers=1602
  papers=97
  papers=64
  papers=86
  papers=8
  papers=109
  papers=1387
  papers=864
  papers=68
  papers=38
  papers=50
  papers=6
  papers=975
  papers=910
  papers=164
  papers=58
  papers=33
  papers=76
  papers=6
  papers=901
  papers=603
  papers=97
  papers=39
  papers=331
  papers=27
  papers=8
  papers=571
  papers=139
  papers=36
  papers=457
  papers=43
  papers=778
  papers=43
  papers=43
  papers=8
  papers=15
Saved ACL all-track papers to: ../Data/acl_dblp_2020_2025_all_tracks.csv
Saved ACL summary to: ../Data/acl_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,ACL,2020,main_proceedings,main_track,778
1,ACL,2020,non-main_proceedings,non-main_track,109
2,ACL,2021,main_proceedings,main_track,710
3,ACL,2021,non-main_proceedings,non-main_track,536
4,ACL,2022,main_proceedings,main_track,700
5,ACL,2022,non-main_proceedings,non-main_track,405
6,ACL,2023,main_proceedings,main_track,1074
7,ACL,2023,non-main_proceedings,non-main_track,1074
8,ACL,2024,main_proceedings,main_track,932
9,ACL,2024,non-main_proceedings,non-main_track,1069


# 3. Collect CVPR

In [3]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CVPR_INDEX_URL = "https://dblp.org/db/conf/cvpr/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "cvpr_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "cvpr_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(CVPR_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/cvpr/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))
    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text



def classify_cvpr_proceedings(dblp_key, proceedings_title):
    """
    CVPR main proceedings use DBLP keys such as conf/cvpr/2025.
    Workshops/challenges/other affiliated volumes usually use suffixes such as
    conf/cvpr/2025w or conf/cvpr/2024medsam.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name, "main_track"

    return "non-main_proceedings", proceedings_name, "non-main_track"


def is_non_paper_title(title):
    text = str(title or "").lower()
    non_paper_patterns = [
        r"\bfront\s*matter\b",
        r"\bfrontmatter\b",
        r"\bpreface\b",
        r"\btable of contents\b",
        r"\bauthor index\b",
        r"\bprogram committee\b",
        r"\borganizing committee\b",
        r"\bproceedings of\b",
        r"\bwelcome message\b",
    ]
    return any(re.search(pattern, text) for pattern in non_paper_patterns)


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])

def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None

def collect_cvpr_proceedings_from_index():
    html_text = request_with_retry(CVPR_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            parsed_year = extract_year_from_h2(tag)
            current_year = parsed_year if parsed_year in YEARS else None
            continue

        if tag.name != "li":
            continue

        if current_year not in YEARS:
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        dblp_key = tag.get("id", "")
        if not dblp_key.startswith("conf/cvpr/"):
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)

        proceedings_type, proceedings_name, track_type = classify_cvpr_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": proceedings_name,
        })

    proceedings_df = pd.DataFrame(proceedings_rows)

    if proceedings_df.empty:
        raise RuntimeError("No CVPR proceedings were found from the DBLP index page.")

    proceedings_df = proceedings_df.drop_duplicates(
        subset=["year", "dblp_key", "toc_url"]
    )

    return proceedings_df.sort_values(
        ["year", "proceedings_type", "track_type", "dblp_key"]
    ).reset_index(drop=True)


def collect_cvpr_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    for li in soup.find_all("li"):
        classes = li.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(li)
        if not title:
            continue

        if is_non_paper_title(title):
            print(f"Skipping non-paper title: {title}")
            continue

        rows.append({
            "conference": "CVPR",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(li),
            "doi": extract_doi(li),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": proceedings_row["track_type"],
            "section/track_name": proceedings_row["section/track_name"],
        })

    return rows


# =========================
# Collect CVPR papers from DBLP
# =========================
proceedings_df = collect_cvpr_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "track_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "track_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting CVPR {row['year']} | {row['proceedings_type']} | "
        f"{row['track_type']} | {row['proceedings_name']}"
    )

    rows = collect_cvpr_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))

cvpr_df = pd.DataFrame(all_rows)

if cvpr_df.empty:
    raise RuntimeError("No CVPR records were collected. Please inspect DBLP page structure.")

cvpr_df = deduplicate(cvpr_df)
cvpr_df = cvpr_df.reindex(columns=STANDARD_COLUMNS)
cvpr_df = cvpr_df.sort_values(
    ["year", "proceedings_type", "track_type", "proceedings_name", "title"]
).reset_index(drop=True)

cvpr_df.to_csv(OUT_CSV, index=False)

summary = (
    cvpr_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved CVPR all-track papers to: {OUT_CSV}")
print(f"Saved CVPR summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,track_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_track,2020 IEEE/CVF Conference on Computer Vision an...,1
1,2020,non-main_proceedings,non-main_track,2020 IEEE/CVF Conference on Computer Vision an...,1
2,2021,main_proceedings,main_track,IEEE Conference on Computer Vision and Pattern...,1
3,2021,non-main_proceedings,non-main_track,IEEE Conference on Computer Vision and Pattern...,1
4,2022,main_proceedings,main_track,IEEE/CVF Conference on Computer Vision and Pat...,1
5,2022,non-main_proceedings,non-main_track,IEEE/CVF Conference on Computer Vision and Pat...,1
6,2023,main_proceedings,main_track,IEEE/CVF Conference on Computer Vision and Pat...,1
7,2023,non-main_proceedings,non-main_track,IEEE/CVF Conference on Computer Vision and Pat...,1
8,2024,main_proceedings,main_track,IEEE/CVF Conference on Computer Vision and Pat...,1
9,2024,non-main_proceedings,non-main_track,IEEE/CVF Conference on Computer Vision and Pat...,1


  papers=1465
  papers=522
  papers=1660
  papers=517
  papers=2072
  papers=560
  papers=2353
  papers=698
  papers=2715
  papers=16
  papers=801
  papers=2871
  papers=13
  papers=659
Saved CVPR all-track papers to: ../Data/cvpr_dblp_2020_2025_all_tracks.csv
Saved CVPR summary to: ../Data/cvpr_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,CVPR,2020,main_proceedings,main_track,1465
1,CVPR,2020,non-main_proceedings,non-main_track,522
2,CVPR,2021,main_proceedings,main_track,1660
3,CVPR,2021,non-main_proceedings,non-main_track,517
4,CVPR,2022,main_proceedings,main_track,2072
5,CVPR,2022,non-main_proceedings,non-main_track,560
6,CVPR,2023,main_proceedings,main_track,2353
7,CVPR,2023,non-main_proceedings,non-main_track,698
8,CVPR,2024,main_proceedings,main_track,2715
9,CVPR,2024,non-main_proceedings,non-main_track,817


# 4. Collect EMNLP

In [4]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMNLP_INDEX_URL = "https://dblp.org/db/conf/emnlp/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "emnlp_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "emnlp_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(EMNLP_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/emnlp/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))

    match = re.search(r"(Proceedings of .*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    match = re.search(r"(Findings of .*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text


def extract_emnlp_year_from_h2_tag(h2_tag):
    h2_id = normalize_text(h2_tag.get("id", ""))
    match = re.search(r"\b(20\d{2})\b", h2_id)
    if match:
        return int(match.group(1))

    text = normalize_text(h2_tag.get_text(" ", strip=True))
    match = re.search(r"\b(20\d{2})\b", text)
    if match:
        return int(match.group(1))

    return None


def classify_emnlp_proceedings(proceedings_title):
    """
    EMNLP is organized as separate proceedings volumes on DBLP.
    Main-track papers are identified at the proceedings-volume level.
    """
    proceedings_name = normalize_text(proceedings_title)
    t = proceedings_name.lower()

    non_main_patterns = [
        "findings of",
        "system demonstrations",
        "industry track",
        "tutorial abstracts",
        "workshop",

    ]

    if any(pattern in t for pattern in non_main_patterns):
        return "non-main_proceedings", proceedings_name, "non-main_track"

    main_patterns = [
        "conference on empirical methods in natural language processing",
    ]

    if any(pattern in t for pattern in main_patterns):
        return "main_proceedings", proceedings_name, "main_track"

    return "non-main_proceedings", proceedings_name, "non-main_track"


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])


def collect_emnlp_proceedings_from_index():
    html_text = request_with_retry(EMNLP_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            year = extract_emnlp_year_from_h2_tag(tag)
            if year is not None:
                current_year = year
            continue

        if tag.name != "li" or current_year not in YEARS:
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)
        proceedings_type, proceedings_name, track_type = classify_emnlp_proceedings(proceedings_title)

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": proceedings_name,
        })

    return pd.DataFrame(proceedings_rows)


def collect_emnlp_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    for li in soup.find_all("li"):
        classes = li.get("class", [])
        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(li)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        rows.append({
            "conference": "EMNLP",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(li),
            "doi": extract_doi(li),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": proceedings_row["track_type"],
            "section/track_name": proceedings_row["section/track_name"],
        })

    return rows


# =========================
# Collect EMNLP papers from DBLP
# =========================
proceedings_df = collect_emnlp_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "track_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "track_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting EMNLP {row['year']} | {row['proceedings_type']} | "
        f"{row['track_type']} | {row['proceedings_name']}"
    )

    rows = collect_emnlp_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))

emnlp_df = pd.DataFrame(all_rows)

if emnlp_df.empty:
    raise RuntimeError("No EMNLP records were collected. Please inspect DBLP page structure.")

emnlp_df = deduplicate(emnlp_df)
emnlp_df = emnlp_df.reindex(columns=STANDARD_COLUMNS)
emnlp_df = emnlp_df.sort_values(
    ["year", "proceedings_type", "track_type", "proceedings_name", "title"]
).reset_index(drop=True)

emnlp_df.to_csv(OUT_CSV, index=False)

summary = (
    emnlp_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved EMNLP all-track papers to: {OUT_CSV}")
print(f"Saved EMNLP summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,track_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_track,Proceedings of the 2020 Conference on Empirica...,1
1,2020,non-main_proceedings,non-main_track,Findings of the Association for Computational ...,1
2,2020,non-main_proceedings,non-main_track,Proceedings of the 1st Workshop on NLP for COV...,1
3,2020,non-main_proceedings,non-main_track,Proceedings of the 2020 Conference on Empirica...,1
4,2020,non-main_proceedings,non-main_track,Proceedings of the 2020 Conference on Empirica...,1
5,2021,main_proceedings,main_track,Proceedings of the 2021 Conference on Empirica...,1
6,2021,non-main_proceedings,non-main_track,Findings of the Association for Computational ...,1
7,2021,non-main_proceedings,non-main_track,Proceedings of the 2021 Conference on Empirica...,1
8,2021,non-main_proceedings,non-main_track,Proceedings of the 2021 Conference on Empirica...,1
9,2022,main_proceedings,main_track,Proceedings of the 2022 Conference on Empirica...,1


  papers=1809
  papers=77
  papers=193
  papers=7
  papers=1405
  papers=1268
  papers=52
  papers=121
  papers=6
  papers=1003
  papers=1047
  papers=6
  papers=52
  papers=77
  papers=1060
  papers=828
  papers=6
  papers=42
  papers=65
  papers=547
  papers=847
  papers=42
  papers=6
  papers=424
  papers=752
  papers=29
  papers=7
  papers=447
  papers=37
Saved EMNLP all-track papers to: ../Data/emnlp_dblp_2020_2025_all_tracks.csv
Saved EMNLP summary to: ../Data/emnlp_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,EMNLP,2020,main_proceedings,main_track,752
1,EMNLP,2020,non-main_proceedings,non-main_track,520
2,EMNLP,2021,main_proceedings,main_track,847
3,EMNLP,2021,non-main_proceedings,non-main_track,472
4,EMNLP,2022,main_proceedings,main_track,828
5,EMNLP,2022,non-main_proceedings,non-main_track,660
6,EMNLP,2023,main_proceedings,main_track,1047
7,EMNLP,2023,non-main_proceedings,non-main_track,1195
8,EMNLP,2024,main_proceedings,main_track,1268
9,EMNLP,2024,non-main_proceedings,non-main_track,1182


# 5. Collect ICLR
dblp中 2020 2021 2022 2025都只有main track, 2023, 2024还多了tiny track

In [5]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ICLR_INDEX_URL = "https://dblp.org/db/conf/iclr/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "iclr_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "iclr_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(ICLR_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/iclr/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))

    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text



def classify_iclr_proceedings(dblp_key, proceedings_title):
    """
    ICLR main proceedings use keys like conf/iclr/2025.
    Non-main volumes, such as Tiny Papers, use suffixes like conf/iclr/2024tiny.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name, "main_track"

    return "non-main_proceedings", proceedings_name, "non-main_track"


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
        "proceedings of"
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])

def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None

def collect_iclr_proceedings_from_index():
    html_text = request_with_retry(ICLR_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            parsed_year = extract_year_from_h2(tag)
            current_year = parsed_year if parsed_year in YEARS else None
            continue

        if tag.name != "li":
            continue

        if current_year not in YEARS:
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        dblp_key = tag.get("id", "")

        if not dblp_key.startswith("conf/iclr/"):
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)

        proceedings_type, proceedings_name, track_type = classify_iclr_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": proceedings_name,
        })

    proceedings_df = pd.DataFrame(proceedings_rows)

    if proceedings_df.empty:
        raise RuntimeError("No ICLR proceedings were found from the DBLP index page.")

    proceedings_df = proceedings_df.drop_duplicates(
        subset=["year", "dblp_key", "toc_url"]
    )

    return proceedings_df.sort_values(
        ["year", "proceedings_type", "track_type", "dblp_key"]
    ).reset_index(drop=True)


def collect_iclr_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    for li in soup.find_all("li"):
        classes = li.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(li)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        rows.append({
            "conference": "ICLR",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(li),
            "doi": extract_doi(li),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": proceedings_row["track_type"],
            "section/track_name": proceedings_row["section/track_name"],
        })

    return rows


# =========================
# Collect ICLR papers from DBLP
# =========================
proceedings_df = collect_iclr_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "track_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "track_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting ICLR {row['year']} | {row['proceedings_type']} | "
        f"{row['track_type']} | {row['proceedings_name']}"
    )

    rows = collect_iclr_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))

iclr_df = pd.DataFrame(all_rows)

if iclr_df.empty:
    raise RuntimeError("No ICLR records were collected. Please inspect DBLP page structure.")

iclr_df = deduplicate(iclr_df)
iclr_df = iclr_df.reindex(columns=STANDARD_COLUMNS)
iclr_df = iclr_df.sort_values(
    ["year", "proceedings_type", "track_type", "proceedings_name", "title"]
).reset_index(drop=True)

iclr_df.to_csv(OUT_CSV, index=False)

summary = (
    iclr_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved ICLR all-track papers to: {OUT_CSV}")
print(f"Saved ICLR summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,track_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_track,8th International Conference on Learning Repre...,1
1,2021,main_proceedings,main_track,9th International Conference on Learning Repre...,1
2,2022,main_proceedings,main_track,The Tenth International Conference on Learning...,1
3,2023,main_proceedings,main_track,The Eleventh International Conference on Learn...,1
4,2023,non-main_proceedings,non-main_track,"The First Tiny Papers Track at ICLR 2023, Tiny...",1
5,2024,main_proceedings,main_track,The Twelfth International Conference on Learni...,1
6,2024,non-main_proceedings,non-main_track,"The Second Tiny Papers Track at ICLR 2024, Tin...",1
7,2025,main_proceedings,main_track,The Thirteenth International Conference on Lea...,1


  papers=687
  papers=860
  papers=1094
  papers=1573
  papers=219
  papers=2260
  papers=194
  papers=3704
Saved ICLR all-track papers to: ../Data/iclr_dblp_2020_2025_all_tracks.csv
Saved ICLR summary to: ../Data/iclr_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,ICLR,2020,main_proceedings,main_track,687
1,ICLR,2021,main_proceedings,main_track,860
2,ICLR,2022,main_proceedings,main_track,1094
3,ICLR,2023,main_proceedings,main_track,1573
4,ICLR,2023,non-main_proceedings,non-main_track,219
5,ICLR,2024,main_proceedings,main_track,2260
6,ICLR,2024,non-main_proceedings,non-main_track,194
7,ICLR,2025,main_proceedings,main_track,3704


# 6. Collect ICML

In [6]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ICML_INDEX_URL = "https://dblp.org/db/conf/icml/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "icml_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "icml_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(ICML_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/icml/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))

    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text


def classify_icml_proceedings(dblp_key, proceedings_title):
    """
    ICML main proceedings use keys like conf/icml/2025.
    Non-main volumes, such as the position paper track, use suffixes like conf/icml/2025p.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name, "main_track"

    return "non-main_proceedings", proceedings_name, "non-main_track"


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
        "proceedings of",
        "beyond explainable artificial intelligence",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])

def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    # Prefer h2 id, e.g. year2025, 2025, conf-sigir-2025
    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    # Fallback to h2 text
    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None

def collect_icml_proceedings_from_index():
    html_text = request_with_retry(ICML_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None
    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            year = extract_year_from_h2(tag)
            if year in YEARS:   
                current_year = year
            else:
                current_year = None
            continue
        if current_year not in YEARS:
            continue
        if tag.name != "li":
            continue
        
        classes = tag.get("class", [])
        if "entry" not in classes:
            continue
        
        dblp_key = tag.get("id", "")

        if not dblp_key.startswith("conf/icml/"):
            continue


        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)
        proceedings_type, proceedings_name, track_type = classify_icml_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": proceedings_name,
        })

    return pd.DataFrame(proceedings_rows)


def collect_icml_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    for li in soup.find_all("li"):
        classes = li.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(li)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        rows.append({
            "conference": "ICML",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(li),
            "doi": extract_doi(li),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": proceedings_row["track_type"],
            "section/track_name": proceedings_row["section/track_name"],
        })

    return rows


# =========================
# Collect ICML papers from DBLP
# =========================
proceedings_df = collect_icml_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "track_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "track_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting ICML {row['year']} | {row['proceedings_type']} | "
        f"{row['track_type']} | {row['proceedings_name']}"
    )

    rows = collect_icml_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))

icml_df = pd.DataFrame(all_rows)

if icml_df.empty:
    raise RuntimeError("No ICML records were collected. Please inspect DBLP page structure.")

icml_df = deduplicate(icml_df)
icml_df = icml_df.reindex(columns=STANDARD_COLUMNS)
icml_df = icml_df.sort_values(
    ["year", "proceedings_type", "track_type", "proceedings_name", "title"]
).reset_index(drop=True)

icml_df.to_csv(OUT_CSV, index=False)

summary = (
    icml_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved ICML all-track papers to: {OUT_CSV}")
print(f"Saved ICML summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,track_type,proceedings_name,n_volumes
0,2020,main_proceedings,main_track,Proceedings of the 37th International Conferen...,1
1,2020,non-main_proceedings,non-main_track,xxAI - Beyond Explainable AI - International W...,1
2,2021,main_proceedings,main_track,Proceedings of the 38th International Conferen...,1
3,2022,main_proceedings,main_track,"International Conference on Machine Learning, ...",1
4,2022,non-main_proceedings,non-main_track,Proceedings of the 1st Workshop on Healthcare ...,1
5,2023,main_proceedings,main_track,"International Conference on Machine Learning, ...",1
6,2024,main_proceedings,main_track,Forty-first International Conference on Machin...,1
7,2025,main_proceedings,main_track,Forty-second International Conference on Machi...,1
8,2025,non-main_proceedings,non-main_track,Forty-second International Conference on Machi...,1
9,2025,non-main_proceedings,non-main_track,Proceedings of The TerraBytes {ICML} Workshop:...,1


  papers=3257
  papers=73
  papers=11
  papers=2610
  papers=1828
  papers=1234
  papers=14
  papers=1183
  papers=1085
  papers=18
Saved ICML all-track papers to: ../Data/icml_dblp_2020_2025_all_tracks.csv
Saved ICML summary to: ../Data/icml_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,ICML,2020,main_proceedings,main_track,1085
1,ICML,2020,non-main_proceedings,non-main_track,18
2,ICML,2021,main_proceedings,main_track,1183
3,ICML,2022,main_proceedings,main_track,1234
4,ICML,2022,non-main_proceedings,non-main_track,14
5,ICML,2023,main_proceedings,main_track,1828
6,ICML,2024,main_proceedings,main_track,2610
7,ICML,2025,main_proceedings,main_track,3257
8,ICML,2025,non-main_proceedings,non-main_track,84


# 7. Collect IJCAI

In [7]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IJCAI_INDEX_URL = "https://dblp.org/db/conf/ijcai/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "ijcai_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "ijcai_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()

    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    candidate_urls = []

    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(IJCAI_INDEX_URL, a["href"])

        if "/db/conf/ijcai/" not in href:
            continue

        if not href.endswith(".html"):
            continue

        if "contents" in label:
            return href

        candidate_urls.append(href)

    if candidate_urls:
        return candidate_urls[0]

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))

    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text


def classify_ijcai_proceedings(dblp_key, proceedings_title):
    """
    IJCAI main proceedings use keys such as conf/ijcai/2025.
    Track type is determined later from h2 headings within each proceedings page.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name

    return "non-main_proceedings", proceedings_name


def is_ijcai_main_track_h2(h2_text):
    text = normalize_text(h2_text).lower()
    return text == "main track"


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])


# =========================
# Collect proceedings from IJCAI index
# =========================
def collect_ijcai_proceedings_from_index():
    html_text = request_with_retry(IJCAI_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            parsed_year = extract_year_from_h2(tag)
            current_year = parsed_year if parsed_year in YEARS else None
            continue

        if tag.name != "li":
            continue

        if current_year not in YEARS:
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        dblp_key = tag.get("id", "")
        if not dblp_key.startswith("conf/ijcai/"):
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)

        proceedings_type, proceedings_name = classify_ijcai_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
        })

    proceedings_df = pd.DataFrame(proceedings_rows)

    if proceedings_df.empty:
        raise RuntimeError("No IJCAI proceedings were found from the DBLP index page.")

    proceedings_df = proceedings_df.drop_duplicates(
        subset=["year", "dblp_key", "toc_url"]
    )

    return proceedings_df.sort_values(
        ["year", "proceedings_type", "dblp_key"]
    ).reset_index(drop=True)


# =========================
# Collect papers from proceedings pages
# =========================
def collect_ijcai_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    current_h2 = ""

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            current_h2 = normalize_text(tag.get_text(" ", strip=True))
            continue

        if tag.name != "li":
            continue

        classes = tag.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(tag)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        if proceedings_row["proceedings_type"] == "main_proceedings":
            track_type = (
                "main_track"
                if is_ijcai_main_track_h2(current_h2)
                else "non-main_track"
            )
            section_track_name = current_h2
        else:
            track_type = "non-main_track"
            section_track_name = current_h2 if current_h2 else proceedings_row["proceedings_name"]

        rows.append({
            "conference": "IJCAI",
            "year": int(proceedings_row["year"]),
            "title": title,
            "authors": extract_authors(tag),
            "doi": extract_doi(tag),
            "source_url": toc_url,
            "proceedings_type": proceedings_row["proceedings_type"],
            "proceedings_name": proceedings_row["proceedings_name"],
            "track_type": track_type,
            "section/track_name": section_track_name,
        })

    return rows


# =========================
# Run collection
# =========================
proceedings_df = collect_ijcai_proceedings_from_index()

print("Proceedings volumes found:")
display(
    proceedings_df
    .groupby(["year", "proceedings_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "proceedings_name"])
)

all_rows = []

for _, row in proceedings_df.iterrows():
    print(
        f"Collecting IJCAI {row['year']} | {row['proceedings_type']} | "
        f"{row['proceedings_name']}"
    )

    rows = collect_ijcai_toc_page(row)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))


ijcai_df = pd.DataFrame(all_rows)

if ijcai_df.empty:
    raise RuntimeError("No IJCAI records were collected. Please inspect DBLP page structure.")

ijcai_df = deduplicate(ijcai_df)
ijcai_df = ijcai_df.reindex(columns=STANDARD_COLUMNS)
ijcai_df = ijcai_df.sort_values(
    ["year", "proceedings_type", "track_type", "section/track_name", "title"]
).reset_index(drop=True)

ijcai_df.to_csv(OUT_CSV, index=False)

summary = (
    ijcai_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved IJCAI all-track papers to: {OUT_CSV}")
print(f"Saved IJCAI summary to: {OUT_SUMMARY}")
display(summary)

Proceedings volumes found:


,year,proceedings_type,proceedings_name,n_volumes
0,2020,main_proceedings,Proceedings of the Twenty-Ninth International ...,1
1,2020,non-main_proceedings,Artificial Intelligence for Knowledge Manageme...,1
2,2020,non-main_proceedings,Human Brain and Artificial Intelligence - Seco...,1
3,2020,non-main_proceedings,Proceedings of AI4Narratives - Workshop on Art...,1
4,2020,non-main_proceedings,Proceedings of the First Workshop on Artificia...,1
5,2021,main_proceedings,Proceedings of the Thirtieth International Joi...,1
6,2021,non-main_proceedings,Artificial Intelligence for Knowledge Manageme...,1
7,2021,non-main_proceedings,Proceedings of the 9th International Workshop ...,1
8,2021,non-main_proceedings,Proceedings of the Second Workshop on Artifici...,1
9,2021,non-main_proceedings,Proceedings of the Twelfth International Works...,1


  papers=778
  papers=5
  papers=15
  papers=11
  papers=11
  papers=721
  papers=4
  papers=15
  papers=13
  papers=7
  papers=863
  papers=9
  papers=10
  papers=8
  papers=11
  papers=19
  papers=9
  papers=846
  papers=7
  papers=1048
  papers=8
  papers=1280
  papers=8
Saved IJCAI all-track papers to: ../Data/ijcai_dblp_2020_2025_all_tracks.csv
Saved IJCAI summary to: ../Data/ijcai_dblp_2020_2025_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,IJCAI,2020,main_proceedings,main_track,592
1,IJCAI,2020,main_proceedings,non-main_track,186
2,IJCAI,2020,non-main_proceedings,non-main_track,42
3,IJCAI,2021,main_proceedings,main_track,586
4,IJCAI,2021,main_proceedings,non-main_track,135
5,IJCAI,2021,non-main_proceedings,non-main_track,39
6,IJCAI,2022,main_proceedings,main_track,679
7,IJCAI,2022,main_proceedings,non-main_track,184
8,IJCAI,2022,non-main_proceedings,non-main_track,66
9,IJCAI,2023,main_proceedings,main_track,639


## 7-2. DBLP中IJCAI 2024的论文缺失DOI，从官网补充

In [8]:
import re
import json
import time
import html
import random
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from difflib import SequenceMatcher


# =========================
# Config
# =========================
BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data")

DBLP_PATH = BASE_DIR / "20260706_dblp/ijcai_dblp_2020_2025_all_tracks.csv"
OUT_DIR = BASE_DIR / "20260712_ijcai_doi_supplement"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IJCAI_2024_URL = "https://www.ijcai.org/proceedings/2024/"
DOI_PREFIX = "10.24963/ijcai.2024"

UPDATED_DBLP_PATH = OUT_DIR / "ijcai_dblp_2020_2025_all_tracks_with_2024_doi.csv"
SUPPLEMENT_REPORT_PATH = OUT_DIR / "ijcai_2024_missing_doi_supplement_report.csv"
OFFICIAL_TITLE_MAP_PATH = OUT_DIR / "ijcai_2024_official_title_doi_map.csv"
UNMATCHED_PATH = OUT_DIR / "ijcai_2024_still_missing_doi_after_official_match.csv"

FUZZY_MATCH = True
FUZZY_THRESHOLD = 0.985


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_title(title):
    title = html.unescape(str(title or "")).lower()
    title = re.sub(r"<[^>]+>", " ", title)
    title = re.sub(r"&amp;", " and ", title)
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "")
    doi = doi.replace("http://dx.doi.org/", "")
    doi = doi.replace("doi:", "")
    doi = doi.strip().rstrip(".")
    if doi in {"", "nan", "none"}:
        return ""
    return doi


def is_missing_doi(doi):
    return normalize_doi(doi) == ""


def request_with_retry(url, n_retry=5, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None

    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=40)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title_from_paper_wrapper(wrapper):
    selectors = [
        ".paper_title",
        ".title",
        "h1",
        "h2",
        "h3",
        "h4",
        "strong",
    ]

    for selector in selectors:
        node = wrapper.select_one(selector)
        if node:
            title = normalize_text(node.get_text(" ", strip=True))
            if title and title.lower() not in {"pdf", "details"}:
                return title

    text_lines = [
        normalize_text(x)
        for x in wrapper.stripped_strings
        if normalize_text(x)
    ]

    skip = {
        "pdf",
        "details",
        "|",
        "(",
        ")",
    }

    for line in text_lines:
        line_clean = line.strip("()| ")
        if not line_clean:
            continue
        if line_clean.lower() in skip:
            continue
        if line_clean.lower().startswith("pdf"):
            continue
        if line_clean.lower().startswith("details"):
            continue

        return line_clean

    return ""


def collect_ijcai_2024_title_doi_map():
    html_text = request_with_retry(IJCAI_2024_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []

    wrappers = soup.find_all(
        "div",
        id=re.compile(r"^paper\d+$"),
        class_=lambda x: x and "paper_wrapper" in x
    )

    if not wrappers:
        wrappers = soup.find_all(
            "div",
            id=re.compile(r"^paper\d+$")
        )

    print(f"Found paper wrappers: {len(wrappers)}")

    for wrapper in wrappers:
        paper_id = wrapper.get("id", "")
        match = re.search(r"paper(\d+)", paper_id)

        if not match:
            continue

        paper_number = int(match.group(1))
        title = extract_title_from_paper_wrapper(wrapper)

        if not title:
            continue

        doi = f"{DOI_PREFIX}/{paper_number}"
        paper_url = f"https://www.ijcai.org/proceedings/2024/{paper_number}"

        rows.append({
            "paper_number": paper_number,
            "official_title": title,
            "official_title_norm": normalize_title(title),
            "doi": doi,
            "doi_url": f"https://doi.org/{doi}",
            "paper_url": paper_url,
        })

    official_df = pd.DataFrame(rows)
    official_df = official_df.drop_duplicates(subset=["paper_number"])
    official_df = official_df.sort_values("paper_number").reset_index(drop=True)

    return official_df


def find_fuzzy_match(title_norm, official_df):
    best_score = 0
    best_row = None

    for _, row in official_df.iterrows():
        official_title_norm = row["official_title_norm"]
        score = SequenceMatcher(None, title_norm, official_title_norm).ratio()

        if score > best_score:
            best_score = score
            best_row = row

    if best_row is not None and best_score >= FUZZY_THRESHOLD:
        return best_row, best_score

    return None, best_score


# =========================
# Step 1. Load DBLP data
# =========================
df = pd.read_csv(DBLP_PATH)

df["conference"] = df["conference"].astype(str).str.upper()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["doi_norm_before"] = df["doi"].map(normalize_doi)
df["title_norm"] = df["title"].map(normalize_title)

target_mask = (
    df["conference"].eq("IJCAI")
    & df["year"].eq(2024)
    & df["doi"].map(is_missing_doi)
)

print(f"IJCAI 2024 papers with missing DOI: {target_mask.sum()}")


# =========================
# Step 2. Parse IJCAI official proceedings
# =========================
official_df = collect_ijcai_2024_title_doi_map()
official_df.to_csv(OFFICIAL_TITLE_MAP_PATH, index=False, encoding="utf-8-sig")

print(f"Official IJCAI 2024 title-DOI map: {len(official_df)}")
print(f"Saved official map to: {OFFICIAL_TITLE_MAP_PATH}")


# =========================
# Step 3. Build title index
# =========================
title_to_rows = {}

for _, row in official_df.iterrows():
    title_norm = row["official_title_norm"]

    if title_norm not in title_to_rows:
        title_to_rows[title_norm] = []

    title_to_rows[title_norm].append(row)


# =========================
# Step 4. Fill missing DOI
# =========================
report_rows = []

for idx, row in df[target_mask].iterrows():
    title = row["title"]
    title_norm = row["title_norm"]

    matched_row = None
    match_method = "unmatched"
    match_score = None
    note = ""

    candidates = title_to_rows.get(title_norm, [])

    if len(candidates) == 1:
        matched_row = candidates[0]
        match_method = "exact_title"
        match_score = 1.0

    elif len(candidates) > 1:
        match_method = "duplicate_exact_title"
        note = f"{len(candidates)} official papers have the same normalized title"

    elif FUZZY_MATCH:
        fuzzy_row, fuzzy_score = find_fuzzy_match(title_norm, official_df)

        if fuzzy_row is not None:
            matched_row = fuzzy_row
            match_method = "fuzzy_title"
            match_score = fuzzy_score
        else:
            match_score = fuzzy_score

    if matched_row is not None:
        doi = matched_row["doi"]

        df.loc[idx, "doi"] = doi
        df.loc[idx, "doi_norm_before"] = row["doi_norm_before"]
        df.loc[idx, "doi_supplement_source"] = "ijcai_official_proceedings"
        df.loc[idx, "ijcai_2024_paper_number"] = matched_row["paper_number"]
        df.loc[idx, "ijcai_2024_paper_url"] = matched_row["paper_url"]
        df.loc[idx, "ijcai_2024_official_title"] = matched_row["official_title"]
        df.loc[idx, "doi_match_method"] = match_method
        df.loc[idx, "doi_match_score"] = match_score
    else:
        df.loc[idx, "doi_supplement_source"] = ""
        df.loc[idx, "ijcai_2024_paper_number"] = ""
        df.loc[idx, "ijcai_2024_paper_url"] = ""
        df.loc[idx, "ijcai_2024_official_title"] = ""
        df.loc[idx, "doi_match_method"] = match_method
        df.loc[idx, "doi_match_score"] = match_score

    report_rows.append({
        "row_index": idx,
        "dblp_title": title,
        "dblp_title_norm": title_norm,
        "old_doi": row.get("doi", ""),
        "new_doi": matched_row["doi"] if matched_row is not None else "",
        "paper_number": matched_row["paper_number"] if matched_row is not None else "",
        "paper_url": matched_row["paper_url"] if matched_row is not None else "",
        "official_title": matched_row["official_title"] if matched_row is not None else "",
        "match_method": match_method,
        "match_score": match_score,
        "note": note,
    })


report_df = pd.DataFrame(report_rows)
report_df.to_csv(SUPPLEMENT_REPORT_PATH, index=False, encoding="utf-8-sig")


# =========================
# Step 5. Save updated DBLP file and unmatched rows
# =========================
df.to_csv(UPDATED_DBLP_PATH, index=False, encoding="utf-8-sig")

still_missing = df[
    df["conference"].eq("IJCAI")
    & df["year"].eq(2024)
    & df["doi"].map(is_missing_doi)
].copy()

still_missing.to_csv(UNMATCHED_PATH, index=False, encoding="utf-8-sig")


# =========================
# Step 6. Summary
# =========================
n_missing_before = int(target_mask.sum())
n_filled = int(report_df["new_doi"].astype(str).str.len().gt(0).sum())
n_still_missing = len(still_missing)

print("=" * 80)
print("IJCAI 2024 DOI supplement summary")
print("=" * 80)
print(f"Missing DOI before: {n_missing_before}")
print(f"Filled DOI:         {n_filled}")
print(f"Still missing DOI:  {n_still_missing}")
print()
print("Match methods:")
print(report_df["match_method"].value_counts(dropna=False))
print()
print(f"Updated DBLP file saved to: {UPDATED_DBLP_PATH}")
print(f"Supplement report saved to: {SUPPLEMENT_REPORT_PATH}")
print(f"Still missing rows saved to: {UNMATCHED_PATH}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/user/GSK/lily/science_of_science/NewDataset/AI_impact_on_cs_publications/Data/20260706_dblp/ijcai_dblp_2020_2025_all_tracks.csv'

# 8. Collect KDD

In [1]:
import html
import re
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
BASE_DIR = Path("/home/user/GSK/lily/science_of_science/NewDataset")
DATA_DIR = BASE_DIR / "AI_impact_on_cs_publications" / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

KDD_INDEX_URL = "https://dblp.org/db/conf/kdd/index.html"
YEARS = range(2020, 2026)

OUT_CSV = DATA_DIR / "kdd_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = DATA_DIR / "kdd_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    )
}


# =========================
# KDD-specific rules
# =========================
MAIN_VOLUME_KEYS = {
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025-1",
    "2025-2",
}

KDD_MAIN_RESEARCH_H2 = {
    (2020, "2020"): {"research track papers"},
    (2021, "2021"): {"research track papers"},
    (2022, "2022"): {"research track full papers"},
    (2023, "2023"): {"research track full papers"},
    (2024, "2024"): {"research track papers"},
    (2025, "2025-1"): {"research track"},
    (2025, "2025-2"): {"research track"},
}


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_heading(text):
    return normalize_text(text).lower()


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    last_error = None

    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "")
    doi = doi.replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
        "proceedings of",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()

        if "doi.org/" in href:
            doi = href.split("doi.org/", 1)[1].strip()
            return normalize_doi_for_key(doi)

        if "doi.acm.org/" in href:
            doi = href.rstrip("/").rsplit("/", 1)[-1].strip()
            return normalize_doi_for_key(doi)

    return ""


def extract_title(li):
    title_node = li.find("span", class_="title")
    if not title_node:
        return ""

    title = normalize_text(title_node.get_text(" ", strip=True))
    return title.rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()

    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(KDD_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/kdd/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))
    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)
    if match:
        return normalize_text(match.group(1))

    return text


def infer_year_from_dblp_key(dblp_key):
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    match = re.match(r"(20\d{2})", tail)
    if match:
        return int(match.group(1))
    return None


def classify_kdd_proceedings(dblp_key, proceedings_title):
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if tail in MAIN_VOLUME_KEYS:
        return "main_proceedings", proceedings_name

    return "non-main_proceedings", proceedings_name


def classify_kdd_track(year, page_year, proceedings_type, h2_text):
    if proceedings_type != "main_proceedings":
        return "non-main_track"

    h2_norm = normalize_heading(h2_text)
    allowed_h2 = KDD_MAIN_RESEARCH_H2.get((year, page_year), set())

    if h2_norm in allowed_h2:
        return "main_track"

    return "non-main_track"


def add_dedup_key(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df


def parse_kdd_index_volumes():
    html_text = request_with_retry(KDD_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    volumes = []

    for li in soup.select("li.entry"):
        dblp_key = li.get("id", "")

        if not dblp_key.startswith("conf/kdd/"):
            continue

        year = infer_year_from_dblp_key(dblp_key)
        if year not in YEARS:
            continue

        toc_url = extract_toc_url_from_proceedings_li(li)
        if not toc_url:
            continue

        page_year = dblp_key.rsplit("/", 1)[-1]
        proceedings_title = extract_proceedings_title(li)
        proceedings_type, proceedings_name = classify_kdd_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        volumes.append({
            "year": year,
            "page_year": page_year,
            "dblp_key": dblp_key,
            "toc_url": toc_url,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
        })

    return pd.DataFrame(volumes)


def collect_kdd_toc_page(volume):
    html_text = request_with_retry(volume["toc_url"])
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    current_h2 = ""

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            current_h2 = normalize_text(tag.get_text(" ", strip=True))
            continue

        if tag.name != "li":
            continue

        classes = tag.get("class", [])
        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(tag)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        track_type = classify_kdd_track(
            year=int(volume["year"]),
            page_year=volume["page_year"],
            proceedings_type=volume["proceedings_type"],
            h2_text=current_h2,
        )

        rows.append({
            "conference": "KDD",
            "year": int(volume["year"]),
            "title": title,
            "authors": extract_authors(tag),
            "doi": extract_doi(tag),
            "source_url": volume["toc_url"],
            "proceedings_type": volume["proceedings_type"],
            "proceedings_name": volume["proceedings_name"],
            "track_type": track_type,
            "section/track_name": current_h2 if current_h2 else volume["proceedings_name"],
        })

    return rows


# =========================
# Run
# =========================
volumes_df = parse_kdd_index_volumes()

print("Proceedings volumes found:")
display(
    volumes_df
    .groupby(["year", "proceedings_type", "proceedings_name"])
    .size()
    .reset_index(name="n_volumes")
    .sort_values(["year", "proceedings_type", "proceedings_name"])
)

all_rows = []

for _, volume in volumes_df.iterrows():
    print(
        f"Collecting KDD {volume['year']} | {volume['page_year']} | "
        f"{volume['proceedings_type']} | {volume['proceedings_name']}"
    )

    rows = collect_kdd_toc_page(volume)
    print(f"  papers={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.6, 1.2))

kdd_df = pd.DataFrame(all_rows)

if kdd_df.empty:
    raise RuntimeError("No KDD records were collected. Please inspect DBLP page structure.")

kdd_df = add_dedup_key(kdd_df)
kdd_df = kdd_df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"]).reset_index(drop=True)
kdd_df = kdd_df.reindex(columns=STANDARD_COLUMNS)
kdd_df = kdd_df.sort_values(
    ["year", "proceedings_type", "track_type", "section/track_name", "title"]
).reset_index(drop=True)

kdd_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

summary = (
    kdd_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(f"  all tracks: {OUT_CSV}")
print(f"  summary: {OUT_SUMMARY}")

display(summary)

Proceedings volumes found:


,year,proceedings_type,proceedings_name,n_volumes
0,2020,main_proceedings,KDD '20: The 26th ACM SIGKDD Conference on Kno...,1
1,2020,non-main_proceedings,Proceedings of the ACM SIGKDD Workshop on Know...,1
2,2020,non-main_proceedings,Proceedings of the KDD 2020 Workshop on Conver...,1
3,2021,main_proceedings,KDD '21: The 27th ACM SIGKDD Conference on Kno...,1
4,2022,main_proceedings,KDD '22: The 28th ACM SIGKDD Conference on Kno...,1
5,2023,main_proceedings,Proceedings of the 29th ACM SIGKDD Conference ...,1
6,2023,non-main_proceedings,Proceedings of EvalRS: A Rounded Evaluation Of...,1
7,2024,main_proceedings,Proceedings of the 30th ACM SIGKDD Conference ...,1
8,2025,main_proceedings,Proceedings of the 31st ACM SIGKDD Conference ...,1
9,2025,main_proceedings,Proceedings of the 31st ACM SIGKDD Conference ...,1


  papers=250
[Retry] 1/4 failed for https://dblp.org/db/conf/kdd/kdd2025-2.html: 503 Server Error: Service Unavailable for url: https://dblp.org/db/conf/kdd/kdd2025-2.html; wait 2.0s
[Retry] 2/4 failed for https://dblp.org/db/conf/kdd/kdd2025-2.html: 503 Server Error: Service Unavailable for url: https://dblp.org/db/conf/kdd/kdd2025-2.html; wait 3.3s
[Retry] 3/4 failed for https://dblp.org/db/conf/kdd/kdd2025-2.html: 503 Server Error: Service Unavailable for url: https://dblp.org/db/conf/kdd/kdd2025-2.html; wait 5.2s
  papers=594
  papers=642
[Retry] 1/4 failed for https://dblp.org/db/conf/kdd/kdd2023.html: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')); wait 2.1s
[Retry] 2/4 failed for https://dblp.org/db/conf/kdd/kdd2023.html: 503 Server Error: Service Unavailable for url: https://dblp.org/db/conf/kdd/kdd2023.html; wait 3.1s
[Retry] 3/4 failed for https://dblp.org/db/conf/kdd/kdd2023.html: 503 Server Error: Service Unavailable for url: https://dblp.org

RuntimeError: Failed to fetch https://dblp.org/db/conf/kdd/kdd2023.html: 503 Server Error: Service Unavailable for url: https://dblp.org/db/conf/kdd/kdd2023.html

## 9. Collect SIGIR
- sigir/index.html中有joint proceedings
- main proceeding中包含main track和其他track，需要根据\<h2\>字段识别
- non-main proceeding是其他会议，所以不要获取了

In [ ]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIGIR_INDEX_URL = "https://dblp.org/db/conf/sigir/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "sigir_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "sigir_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# SIGIR h2 rules
# =========================
SIGIR_MAIN_H2 = {
    2019: [
        "Session 1A: Learning to Rank 1",
        "Session 1B: Health and Social Media",
        "Session 1C: Search Intents",
        "Session 2A: Question Answering",
        "Session 2B: Collaborative Filtering",
        "Session 2C: Knowledge and Entities",
        "Session 3A: Recommendations 1",
        "Session 3B: Interpretatibility and Explainability",
        "Session 3C: Fact-checking, Privacy and Legal",
        "Session 4A: Recommendations and Classificatiion",
        "Session 4B: Queries",
        "Session 4C: Users and Tasks",
        "Session 5A: Conversation and Dialog",
        "Session 5B: Efficiency, Effectiveness and Performance",
        "Session 6A: Social Media",
        "Session 6B: Personalization and Personal Data Search",
        "Session 7A: Relevance and Evaluation 1",
        "Session 7B: Multilingual and Cross-modal Retrieval",
        "Session 7C: Recommendations 2",
        "Session 8A: User Behavior and Experience",
        "Session 8B: Hashing",
        "Session 8C: Summarization and Information Extraction",
        "Session 9A: Fashion Match",
        "Session 9B: Relevance and Evaluation 2",
        "Session 9C: Learning to Rank 2",
        "Short Research Papers 1A: AI, Mining, and Others",
        "Short Research Papers 1B: Recommendation and Evaluation",
        "Short Research Papers 1C: Search",
        "Short Research Papers 2A: AI, Mining, and Others",
        "Short Research Papers 2B: Recommendation and Evaluation",
        "Short Research Papers 2C: Search",
        "Short Research Papers 3A: AI, Mining, and Others",
        "Short Research Papers 3B: Recommendation and Evaluation",
        "Short Research Papers 3C: Search"
    ],
    2020: [
        "Session 1A: NeuIR and Semantic Matching",
        "Session 1B: Knowledge and Explainability",
        "Session 1C: Graph-based Analysis",
        "Session 2A: Knowledge for Personalization",
        "Session 2B: User Behavior and Experience",
        "Session 2C: Evaluation",
        "Session 3A: Bias and Fairness",
        "Session 3B: Learning to Rank",
        "Session 3C: Question Answering",
        "Session 4A: Query and Representation",
        "Session 4B: Graph-based Recommendation",
        "Session 4C: Neural Networks and Embedding",
        "Session 5A: Domain Specific Applications 1",
        "Session 5B: Learning for Recommendation",
        "Session 5C: Information Access and Filtering",
        "Session 6A: Neural Collaborative Filtering 1",
        "Session 6B: Domain Specific Applications 2",
        "Session 6C: Context-aware Modeling",
        "Session 7A: Conversation and Interactive IR",
        "Session 7B: Text Classification and Transfer Learning",
        "Session 7C: Neural Collaborative Filtering 2",
        "Session 8A: Domain Specific Retrieval Tasks",
        "Session 8B: Multi-modal Retrieval and Ranking",
        "Session 8C: Sequential Recommendation",
        "Short Research Papers I",
        "Short Research Papers II",
    ],
    2021: [
        "Session 1A: Bias and counterfactual learning 1",
        "Session 1B: Recommendation 1",
        "Session 1C: Searching and Ranking",
        "Session 1D: Social Aspects",
        "Session 1E: Knowledge Structures",
        "Session 1F: Applications 1",
        "Session 2A: Bias and Counterfactual Learning 2",
        "Session 2B: Recommendation 2",
        "Session 2C: Sequences and Sessions",
        "Session 2D: Time Matters",
        "Session 2E: Question Answering",
        "Session 2F: Applications 2",
        "Session 3A: Conversational IR 1",
        "Session 3B: Recommendation 3",
        "Session 3C: Neural IR",
        "Session 3D: Cross-domain IR",
        "Session 3E: Diversity and Novelty",
        "Session 3F: Applications 3",
        "Session 4A: Conversational IR 2",
        "Session 4B: Recommendation 4",
        "Session 4C: Learning to Rank",
        "Session 4D: Legal IR",
        "Session 4E: Fairness",
        "Session 4F: Adversarial IR",
        "Session 5A: Multi-modal IR",
        "Session 5B: Exploration and Cold Start",
        "Session 5C: Mining and Classification",
        "Session 5D: Click Models and Prediction",
        "Session 5E: Efficiency",
        "Session 6A: Multimedia IR",
        "Session 6B: Reinforcement Learning and Bandits",
        "Session 6C: Natural Language and Semantics",
        "Session 6D: IR Models",
        "Session 6E: Evaluation",
        "Short Research Papers I",
        "Short Research Papers II",
        "Short Research Papers III",
    ],
    2022: [
        *[f"Topic {i}: " for i in range(1, 24)],
        "Short Research Papers",
    ],
    2023: [
        *[f"Session {i} -" for i in range(1, 34)],
        "Short Research Papers",
    ],
    2024: [
        "Session: LLMs and Search",
        "Session: Reasoning and Knowledge Graphs",
        "Session: Efficiency for Search",
        "Session: Multimedia 1",
        "Session: Evaluation",
        "Session: RecSys and LLMs",
        "Session: Fairness in RecSys",
        "Session: GenIR and The Future of Search with LLMs",
        "Session: Graphs and LLMs",
        "Session: Domain Specific RecSys",
        "Session: Multilingual Retrieval",
        "Session: NLP",
        "Session: Multimodal RecSys",
        "Session: Retrieval Augmented Generation",
        "Session: Conversational IR and Recommendation",
        "Session: Multimodal",
        "Session: Graphs and RecSys 1",
        "Session: Recommendation Systems",
        "Session: Users and Simulations",
        "Session: Explanability in Search and Recommendation",
        "Domain Specific",
        "CTR, Ads and Click Models",
        "Session: Graphs and RecSys 2",
        "Session: Dense Retrieval 1",
        "Session: Diffusion in RecSys",
        "Session: Neural IR",
        "Session: Point-of-Interest Recommendation",
        "Session: Fairness",
        "Session: Sequential Recommendation",
        "Session: Networks and Graphs",
        "Session: Privacy, Security and Federated Learning",
        "Session: Prompts, Instructions and LLMs in Recommender Systems",
        "Session: Dense Retrieval 2",
        "Session: Long-term and Session Recommendation",
        "Session: Evaluation with and for LLMs",
        "Session: Question Answering and Summarisation",
        "Session: Cross-Domain Recommendation",
        "Session: Multimedia 2",
        "Session: Legal",
        "Session: Short Research Papers",
    ],
    2025: [
        "Conversational IR and Intelligent Agents",
        "Benchmarks and Datasets",
        "Evaluation",
        "Domain-specific Applications 1",
        "Domain-specific Applications 2",
        "FATE 1",
        "FATE 2",
        "FATE 3",
        "Humans and Interfaces",
        "Machine Learning 1",
        "Machine Learning 2",
        "Image Retrieval",
        "Video Retrieval",
        "Multi-modal Retrieval",
        "Biomedical and Health",
        "Question Answering",
        "Knowledge and Knowledge Graphs",
        "Natural Language Processing 1",
        "Natural Language Processing 2",
        "Natural Language Processing 3",
        "RecSys: Sequential 1",
        "RecSys: Sequential 2",
        "RecSys: Sequential 3",
        "RecSys: FATE",
        "RecSys: Domain-specific",
        "RecSys: Multimodal",
        "RecSys: LLMs",
        "RecSys: Collaborative Filtering",
        "RecSys: Graphs",
        "RecSys: Scalability, Embeddings and Training",
        "RecSys: Ranking and Adaptivity",
        "Reranking",
        "Search and Ranking 1",
        "Search and Ranking 2",
        "Efficiency",
        "Short Research Papers",
    ],
}


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(SIGIR_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/sigir/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))
    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)

    if match:
        return normalize_text(match.group(1))

    return text


def classify_sigir_proceedings(dblp_key, proceedings_title):
    """
    SIGIR main proceedings use DBLP keys like conf/sigir/2025.
    Other SIGIR-related volumes use suffixes such as conf/sigir/2022inra.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name

    return "non-main_proceedings", proceedings_name


def is_main_h2(year, h2_text):
    h2_text = normalize_text(h2_text)
    rules = SIGIR_MAIN_H2.get(year, [])

    for rule in rules:
        if year in {2022, 2023} and h2_text.startswith(rule):
            return True
        if h2_text == rule:
            return True

    return False


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])

def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    # Prefer h2 id, e.g. year2025, 2025, conf-sigir-2025
    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    # Fallback to h2 text
    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None

# =========================
# Collect proceedings from SIGIR index
# =========================
def collect_sigir_proceedings_from_index():
    html_text = request_with_retry(SIGIR_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            year = extract_year_from_h2(tag)
            if year in YEARS:
                current_year = year
            else:
                current_year = None
            continue

        if tag.name != "li":
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        dblp_key = tag.get("id", "")

        if not dblp_key.startswith("conf/sigir/"):
            continue

        if current_year not in YEARS:
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)

        proceedings_type, proceedings_name = classify_sigir_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
        })

    proceedings_df = pd.DataFrame(proceedings_rows)

    if proceedings_df.empty:
        raise RuntimeError("No SIGIR proceedings were found from the DBLP index page.")

    return proceedings_df.sort_values(
        ["year", "proceedings_type", "dblp_key"]
    ).reset_index(drop=True)


# =========================
# Collect papers from each proceedings page
# =========================
def collect_sigir_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    year = int(proceedings_row["year"])
    proceedings_type = proceedings_row["proceedings_type"]
    proceedings_name = proceedings_row["proceedings_name"]

    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    current_h2 = ""

    for tag in soup.find_all(["h2", "li"]):

        if tag.name == "h2":
            current_h2 = normalize_text(tag.get_text(" ", strip=True))
            continue
        if current_h2 == "Keynote & Invited Talks":
            continue
        if tag.name != "li":
            continue

        classes = tag.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(tag)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        if proceedings_type == "main_proceedings":
            track_type = "main_track" if is_main_h2(year, current_h2) else "non-main_track"
            section_track_name = current_h2
        else:
            track_type = "non-main_track"
            section_track_name = current_h2 if current_h2 else proceedings_name

        rows.append({
            "conference": "SIGIR",
            "year": year,
            "title": title,
            "authors": extract_authors(tag),
            "doi": extract_doi(tag),
            "source_url": toc_url,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": section_track_name,
        })

    return rows


# =========================
# Run collection
# =========================
proceedings_df = collect_sigir_proceedings_from_index()

print("Collected SIGIR proceedings from DBLP index:")
display(proceedings_df[["year", "dblp_key", "proceedings_type", "proceedings_name", "toc_url"]])

all_rows = []

for _, proceedings_row in proceedings_df.iterrows():
    print(
        f"Collecting SIGIR {proceedings_row['year']} | "
        f"{proceedings_row['proceedings_type']} | "
        f"{proceedings_row['dblp_key']}"
    )

    rows = collect_sigir_toc_page(proceedings_row)
    print(f"  collected={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))


sigir_df = pd.DataFrame(all_rows)

if sigir_df.empty:
    raise RuntimeError("No SIGIR records were collected. Please inspect DBLP page structure.")

sigir_df = deduplicate(sigir_df)
sigir_df = sigir_df.reindex(columns=STANDARD_COLUMNS)
sigir_df = sigir_df.sort_values(
    ["year", "proceedings_type", "track_type", "section/track_name", "title"]
).reset_index(drop=True)

sigir_df.to_csv(OUT_CSV, index=False)

summary = (
    sigir_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved SIGIR all-track papers to: {OUT_CSV}")
print(f"Saved SIGIR summary to: {OUT_SUMMARY}")
display(summary)

Collected SIGIR proceedings from DBLP index:


,year,dblp_key,proceedings_type,proceedings_name,toc_url
0,2019,conf/sigir/2019,main_proceedings,Proceedings of the 42nd International ACM SIGI...,https://dblp.org/db/conf/sigir/sigir2019.html
1,2019,conf/sigir/2019birndl,non-main_proceedings,Proceedings of the 4th Joint Workshop on Bibli...,https://dblp.org/db/conf/sigir/birndl2019.html
2,2019,conf/sigir/2019osirrc,non-main_proceedings,Proceedings of the Open-Source IR Replicabilit...,https://dblp.org/db/conf/sigir/osirrc2019.html


  collected=260
  collected=27
  collected=14
Saved SIGIR all-track papers to: ../Train_Data/sigir_dblp_2019_all_tracks.csv
Saved SIGIR summary to: ../Train_Data/sigir_dblp_2019_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,SIGIR,2019,main_proceedings,main_track,192
1,SIGIR,2019,main_proceedings,non-main_track,68
2,SIGIR,2019,non-main_proceedings,non-main_track,41


## 10. 根据h2筛选WWW主会文章

In [ ]:
import re
import html
import time
import random
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =========================
# Config
# =========================
OUT_DIR = Path("../Data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

WWW_INDEX_URL = "https://dblp.org/db/conf/www/index.html"
YEARS = range(2020, 2026)

OUT_CSV = OUT_DIR / "www_dblp_2020_2025_all_tracks.csv"
OUT_SUMMARY = OUT_DIR / "www_dblp_2020_2025_summary.csv"

STANDARD_COLUMNS = [
    "conference", "year", "title", "authors", "doi",
    "source_url", "proceedings_type", "proceedings_name",
    "track_type", "section/track_name"
]


# =========================
# Helpers
# =========================
def normalize_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def request_with_retry(url, n_retry=4, sleep_base=1.5):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0 Safari/537.36"
        )
    }

    last_error = None
    for attempt in range(1, n_retry + 1):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            wait = sleep_base * attempt + random.uniform(0, 0.8)
            print(f"[Retry] {attempt}/{n_retry} failed for {url}: {exc}; wait {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {url}: {last_error}")


def extract_title(li):
    node = li.find("span", class_="title")
    if node is None:
        return ""
    return normalize_text(node.get_text(" ", strip=True)).rstrip(".")


def extract_authors(li):
    authors = []

    for node in li.select('[itemprop="author"] [itemprop="name"]'):
        name = normalize_text(node.get_text(" ", strip=True))
        if name:
            authors.append(name)

    if not authors:
        for a in li.find_all("a", href=True):
            href = a["href"]
            if "/pid/" in href:
                name = normalize_text(a.get_text(" ", strip=True))
                if name:
                    authors.append(name)

    out = []
    seen = set()
    for name in authors:
        if name not in seen:
            seen.add(name)
            out.append(name)

    return "; ".join(out)


def extract_doi(li):
    for a in li.find_all("a", href=True):
        href = a["href"].strip()
        if "doi.org/" in href:
            return href.split("doi.org/", 1)[1].strip().lower().rstrip(".")
    return ""


def extract_toc_url_from_proceedings_li(li):
    for a in li.find_all("a", href=True):
        label = normalize_text(a.get_text(" ", strip=True)).lower()
        href = urljoin(WWW_INDEX_URL, a["href"])

        if "contents" in label and "/db/conf/www/" in href and href.endswith(".html"):
            return href

    return ""


def extract_proceedings_title(li):
    title = extract_title(li)
    if title:
        return title

    text = normalize_text(li.get_text(" ", strip=True))
    match = re.search(r"(.*?)(?:\s+\[contents\]|\s+dblp key:|\s+view)", text)

    if match:
        return normalize_text(match.group(1))

    return text



def classify_www_proceedings(dblp_key, proceedings_title):
    """
    WWW main proceedings use DBLP keys like conf/www/2025.
    Companion, workshop, and other co-located volumes use suffixes such as
    conf/www/2025c or conf/www/2021cleopatra.
    """
    tail = str(dblp_key or "").rsplit("/", 1)[-1]
    proceedings_name = normalize_text(proceedings_title)

    if re.fullmatch(r"20\d{2}", tail):
        return "main_proceedings", proceedings_name

    return "non-main_proceedings", proceedings_name


def is_www_main_track_h2(year, h2_text):
    text = normalize_text(h2_text)
    text_lower = text.lower()

    if not text_lower:
        return False

    non_main_keywords = [
        "keynote",
        "tutorial",
        "panel",
        "demo",
        "demonstration",
        "doctoral",
        "workshop",
        "companion",
        "challenge",
    ]

    if any(k in text_lower for k in non_main_keywords):
        return False

    # WWW 2020 has a compact section structure.
    # Full papers and short papers are treated as main-track papers.
    # "Future of the Web Track" is kept as non-main_track.
    if year == 2020:
        return text_lower in {
            "session: full paper",
            "session: short paper",
        }

    # WWW 2021 uses topical "Session: ..." headings for main papers.
    if year == 2021:
        return text_lower.startswith("session:")

    # WWW 2022 explicitly labels research-track sessions.
    if year == 2022:
        return "research track:" in text_lower

    # WWW 2023 uses topical headings without the "Research Track" prefix.
    # After removing keynotes above, the remaining topical sections are main-track sections.
    if year == 2023:
        return True

    # WWW 2024 explicitly labels research-track sections.
    if year == 2024:
        return text_lower.startswith("research track:")

    # WWW 2025 uses numbered oral sessions and poster sessions for main papers.
    if year == 2025:
        if re.match(r"^session\s+\d+\s*:", text_lower):
            return True
        if re.match(r"^poster session\s+\d+", text_lower):
            return True
        return False

    return False


def is_non_paper_title(title):
    text = normalize_text(title).lower().rstrip(".")

    exact_non_paper_titles = {
        "front matter",
        "frontmatter",
        "preface",
        "table of contents",
        "author index",
        "program committee",
        "organizing committee",
        "welcome message",
    }

    if text in exact_non_paper_titles:
        return True

    if text.startswith("proceedings of "):
        return True

    return False


def normalize_title_for_key(title):
    title = str(title or "").lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def normalize_doi_for_key(doi):
    doi = str(doi or "").strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("https://dx.doi.org/", "").replace("http://dx.doi.org/", "")
    return doi.strip().rstrip(".")


def deduplicate(df):
    df = df.copy()

    doi_key = df["doi"].fillna("").map(normalize_doi_for_key)
    title_key = df["title"].fillna("").map(normalize_title_for_key)

    df["_dedup_key"] = [
        f"{conf}|{year}|doi|{doi}" if doi else f"{conf}|{year}|title|{title}"
        for conf, year, doi, title in zip(
            df["conference"], df["year"], doi_key, title_key
        )
    ]

    return df.drop_duplicates("_dedup_key").drop(columns=["_dedup_key"])

def extract_year_from_h2(h2):
    h2_id = str(h2.get("id", ""))
    h2_text = normalize_text(h2.get_text(" ", strip=True))

    # Prefer h2 id, e.g. year2025, 2025, conf-sigir-2025
    match = re.search(r"(20\d{2})", h2_id)
    if match:
        return int(match.group(1))

    # Fallback to h2 text
    match = re.search(r"(20\d{2})", h2_text)
    if match:
        return int(match.group(1))

    return None

# =========================
# Collect proceedings from WWW index
# =========================
def collect_www_proceedings_from_index():
    html_text = request_with_retry(WWW_INDEX_URL)
    soup = BeautifulSoup(html_text, "html.parser")

    proceedings_rows = []
    current_year = None

    for tag in soup.find_all(["h2", "li"]):
        if tag.name == "h2":
            year = extract_year_from_h2(tag)
            if year in YEARS:
                current_year = year
            else:
                current_year = None
            continue

        if tag.name != "li":
            continue

        classes = tag.get("class", [])
        if "entry" not in classes:
            continue

        dblp_key = tag.get("id", "")

        if not dblp_key.startswith("conf/www/"):
            continue

        if current_year not in YEARS:
            continue

        toc_url = extract_toc_url_from_proceedings_li(tag)
        if not toc_url:
            continue

        proceedings_title = extract_proceedings_title(tag)

        proceedings_type, proceedings_name = classify_www_proceedings(
            dblp_key=dblp_key,
            proceedings_title=proceedings_title,
        )

        proceedings_rows.append({
            "year": current_year,
            "toc_url": toc_url,
            "dblp_key": dblp_key,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
        })

    proceedings_df = pd.DataFrame(proceedings_rows)

    if proceedings_df.empty:
        raise RuntimeError("No WWW proceedings were found from the DBLP index page.")

    return proceedings_df.sort_values(
        ["year", "proceedings_type", "dblp_key"]
    ).reset_index(drop=True)


# =========================
# Collect papers from each proceedings page
# =========================
def collect_www_toc_page(proceedings_row):
    toc_url = proceedings_row["toc_url"]
    year = int(proceedings_row["year"])
    proceedings_type = proceedings_row["proceedings_type"]
    proceedings_name = proceedings_row["proceedings_name"]

    html_text = request_with_retry(toc_url)
    soup = BeautifulSoup(html_text, "html.parser")

    rows = []
    current_h2 = ""

    for tag in soup.find_all(["h2", "li"]):

        if tag.name == "h2":
            current_h2 = normalize_text(tag.get_text(" ", strip=True))
            continue
        if current_h2 == "Keynote Talk":
            continue
        if tag.name != "li":
            continue

        classes = tag.get("class", [])

        if "entry" not in classes or "inproceedings" not in classes:
            continue

        title = extract_title(tag)
        if not title:
            continue

        if is_non_paper_title(title):
            continue

        if proceedings_type == "main_proceedings":
            track_type = (
                "main_track"
                if is_www_main_track_h2(year, current_h2)
                else "non-main_track"
            )
            section_track_name = current_h2
        else:
            track_type = "non-main_track"
            section_track_name = current_h2 if current_h2 else proceedings_name

        rows.append({
            "conference": "WWW",
            "year": year,
            "title": title,
            "authors": extract_authors(tag),
            "doi": extract_doi(tag),
            "source_url": toc_url,
            "proceedings_type": proceedings_type,
            "proceedings_name": proceedings_name,
            "track_type": track_type,
            "section/track_name": section_track_name,
        })

    return rows


# =========================
# Run collection
# =========================
proceedings_df = collect_www_proceedings_from_index()

print("Collected WWW proceedings from DBLP index:")
display(proceedings_df[["year", "dblp_key", "proceedings_type", "proceedings_name", "toc_url"]])

all_rows = []

for _, proceedings_row in proceedings_df.iterrows():
    print(
        f"Collecting WWW {proceedings_row['year']} | "
        f"{proceedings_row['proceedings_type']} | "
        f"{proceedings_row['dblp_key']}"
    )

    rows = collect_www_toc_page(proceedings_row)
    print(f"  collected={len(rows)}")

    all_rows.extend(rows)
    time.sleep(random.uniform(0.4, 0.9))


www_df = pd.DataFrame(all_rows)

if www_df.empty:
    raise RuntimeError("No WWW records were collected. Please inspect DBLP page structure.")

www_df = deduplicate(www_df)
www_df = www_df.reindex(columns=STANDARD_COLUMNS)
www_df = www_df.sort_values(
    ["year", "proceedings_type", "track_type", "section/track_name", "title"]
).reset_index(drop=True)

www_df.to_csv(OUT_CSV, index=False)

summary = (
    www_df.groupby(["conference", "year", "proceedings_type", "track_type"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "year", "proceedings_type", "track_type"])
)

summary.to_csv(OUT_SUMMARY, index=False)

print(f"Saved WWW all-track papers to: {OUT_CSV}")
print(f"Saved WWW summary to: {OUT_SUMMARY}")
display(summary)

Collected WWW proceedings from DBLP index:


,year,dblp_key,proceedings_type,proceedings_name,toc_url
0,2019,conf/www/2019,main_proceedings,"The World Wide Web Conference, WWW 2019, San F...",https://dblp.org/db/conf/www/www2019.html
1,2019,conf/www/2019c,non-main_proceedings,Companion of The 2019 World Wide Web Conferenc...,https://dblp.org/db/conf/www/www2019c.html


  collected=388
  collected=252
Saved WWW all-track papers to: ../Train_Data/www_dblp_2019_all_tracks.csv
Saved WWW summary to: ../Train_Data/www_dblp_2019_summary.csv


,conference,year,proceedings_type,track_type,n_papers
0,WWW,2019,main_proceedings,non-main_track,388
1,WWW,2019,non-main_proceedings,non-main_track,252


# 统计各个会议每年的main track和non-main track的数量

In [2]:
import pandas as pd

aaai = pd.read_csv("../Data/20260706/aaai_dblp_2020_2025_summary.csv")
acl = pd.read_csv("../Data/20260706/acl_dblp_2020_2025_summary.csv")
cvpr = pd.read_csv("../Data/20260706/cvpr_dblp_2020_2025_summary.csv")
emnlp = pd.read_csv("../Data/20260706/emnlp_dblp_2020_2025_summary.csv")
iclr = pd.read_csv("../Data/20260706/iclr_dblp_2020_2025_summary.csv")
icml = pd.read_csv("../Data/20260706/icml_dblp_2020_2025_summary.csv")
ijcai = pd.read_csv("../Data/20260706/ijcai_dblp_2020_2025_summary.csv")
kdd = pd.read_csv("../Data/20260706/kdd_dblp_2020_2025_summary.csv")
sigir = pd.read_csv("../Data/20260706/sigir_dblp_2020_2025_summary.csv")
www = pd.read_csv("../Data/20260706/www_dblp_2020_2025_summary.csv")

tracks_summary = pd.concat([aaai, acl, cvpr, emnlp, iclr, icml, ijcai, kdd, sigir, www], ignore_index=True)

summary = tracks_summary.groupby(["conference", "year", "track_type"], as_index=False)["n_papers"].sum().sort_values(
    ["conference", "year", "track_type"]
)

summary.to_csv("../Data/20260706/all_conferences_2020_2025_summary.csv", index=False, encoding="utf-8-sig")

display(summary)

,conference,year,track_type,n_papers
0,AAAI,2020,main_track,1584
1,AAAI,2020,non_main_track,321
2,AAAI,2021,main_track,1654
3,AAAI,2021,non_main_track,409
4,AAAI,2022,main_track,1319
...,...,...,...,...
108,WWW,2023,non-main_track,297
109,WWW,2024,main_track,405
110,WWW,2024,non-main_track,373
111,WWW,2025,main_track,409


In [2]:
import pandas as pd

summary = pd.read_csv("../Data/20260706/all_conferences_2020_2025_summary.csv")

sum = summary['n_papers'].sum()
print(sum)

91890
